# Triton vector add - verification lab

Read `tutorial.md` first. This notebook asks for predictions, then gathers runnable evidence. It is intended for WSL2 with an NVIDIA GPU; do not infer GPU performance from a Mac/MPS run.

Learning order: preflight -> predict -> correctness including a tail block -> bandwidth sweep -> write `note.md`.

In [ ]:
# Environment evidence. Record this output in your experiment note.
import platform
import sys

import torch
import triton
import triton.language as tl

print('python:', sys.version.split()[0])
print('platform:', platform.platform())
print('torch:', torch.__version__)
print('triton:', triton.__version__)
print('cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Run this lab on the WSL2 NVIDIA GPU environment.'
print('gpu:', torch.cuda.get_device_name())
print('cuda runtime:', torch.version.cuda)

## Predict before running

For `N=1025` and `BLOCK_SIZE=1024`, write your answer here before you execute the kernel:

- grid size:
- offsets for `pid=1`:
- valid entries in that block's mask:
- why a benchmark needs synchronization:


In [ ]:
@triton.jit
def add_kernel(x_ptr, y_ptr, z_ptr, n_elements, BLOCK_SIZE: tl.constexpr):
    # One logical program owns one contiguous block of output indices.
    pid = tl.program_id(axis=0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    # The final program may contain indices beyond n_elements.
    mask = offsets < n_elements
    x = tl.load(x_ptr + offsets, mask=mask, other=0.0)
    y = tl.load(y_ptr + offsets, mask=mask, other=0.0)
    tl.store(z_ptr + offsets, x + y, mask=mask)


def triton_add(x: torch.Tensor, y: torch.Tensor, block_size: int = 1024) -> torch.Tensor:
    assert x.is_cuda and y.is_cuda
    assert x.shape == y.shape and x.dtype == y.dtype
    assert x.is_contiguous() and y.is_contiguous()
    z = torch.empty_like(x)
    n_elements = z.numel()
    grid = (triton.cdiv(n_elements, block_size),)
    add_kernel[grid](x, y, z, n_elements, BLOCK_SIZE=block_size)
    return z

In [ ]:
# Correctness first: tail sizes exercise the mask without unsafe out-of-bounds experiments.
torch.manual_seed(0)
block_size = 1024
for n in [1, block_size - 1, block_size, block_size + 1, 1_000_003]:
    x = torch.randn(n, device='cuda', dtype=torch.float32)
    y = torch.randn(n, device='cuda', dtype=torch.float32)
    actual = triton_add(x, y, block_size)
    expected = torch.add(x, y)
    torch.cuda.synchronize()  # Make an asynchronous error visible here.
    max_error = (actual - expected).abs().max().item()
    print(f'N={n:>8}, grid={triton.cdiv(n, block_size):>4}, max_error={max_error:.1e}')
    assert torch.equal(actual, expected)

## Tail-mask evidence

Do not delete the mask to create a failure: an out-of-bounds GPU access is undefined and can destabilize the process. Instead, use the `N=1025` result above as positive evidence that the final program safely writes exactly one valid element. State the coverage invariant in `note.md`.

In [ ]:
# Kernel-only timing: allocations are outside the timed region; CUDA events create explicit GPU timing boundaries.
import statistics


def median_ms(fn, warmup: int = 25, repeats: int = 100) -> float:
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    samples = []
    for _ in range(repeats):
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)
        start.record()
        fn()
        end.record()
        end.synchronize()
        samples.append(start.elapsed_time(end))
    return statistics.median(samples)


n = 2**24
x = torch.randn(n, device='cuda', dtype=torch.float32)
y = torch.randn(n, device='cuda', dtype=torch.float32)
reference_out = torch.empty_like(x)
bytes_moved = 3 * n * x.element_size()

for block_size in [256, 512, 1024, 2048]:
    ms = median_ms(lambda: triton_add(x, y, block_size))
    gbps = bytes_moved / (ms * 1e-3) / 1e9
    print(f'triton block={block_size:>4}: {ms:7.3f} ms, {gbps:8.1f} effective GB/s')

torch_ms = median_ms(lambda: torch.add(x, y, out=reference_out))
torch_gbps = bytes_moved / (torch_ms * 1e-3) / 1e9
print(f'torch out= kernel: {torch_ms:7.3f} ms, {torch_gbps:8.1f} effective GB/s')

## Record before interpreting

Copy the following into `note.md` after the run:

- Hardware and software versions:
- Prediction versus observed tail-block result:
- Block-size sweep table with units:
- Why the measurement supports or does not support a memory-bandwidth bottleneck:
- What this benchmark does **not** prove about other kernels or other GPUs:
- One question to carry into tiled matmul:
